# Workshop 2 — Face Analysis (age / gender / ethnicity)

This notebook builds on detection and runs demographic analysis with `FaceAnalyzer`. It annotates each face crop with predictions and shows a labeled grid.

**Note:** `FaceAnalyzer` will lazy-load models (InsightFace & optional FairFace).

In [ ]:
# Ensure project path is importable
import os, sys
proj_root = os.path.abspath(os.path.join('..', 'lib'))
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

import cv2, numpy as np, matplotlib.pyplot as plt
from face_detector import FaceDetector, select_image_file
from face_analyzer import FaceAnalyzer

detector = FaceDetector()
analyzer = FaceAnalyzer()

IMAGE_PATH = None  # set to a path to skip dialog
if IMAGE_PATH is None:
    try:
        IMAGE_PATH = select_image_file()
    except Exception:
        pass
if not IMAGE_PATH:
    raise RuntimeError('No image selected. Set IMAGE_PATH to a local image path.')

In [ ]:
img = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
faces = detector.process_frame(img)
print(f'Found {len(faces)} faces')

# Display the original image before drawing detections
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
# Analyze each face and produce annotated thumbnails
thumbs = []
for i, face in enumerate(faces):
    face_img = face['face_img']
    # FaceAnalyzer expects BGR cropping (we already have it)
    attrs = analyzer.analyze_face(face_img)
    print(f"Face {i+1} attributes: {attrs}")
    # Prepare a display thumbnail (RGB)
    disp = cv2.resize(face_img, (180,180))
    disp = cv2.cvtColor(disp, cv2.COLOR_BGR2RGB)
    label = f"{i+1}: {attrs.get('gender','?')} {attrs.get('age','?')}y\n{attrs.get('ethnicity','?')}"
    thumbs.append((disp, label))
    
# Show grid with labels
cols = min(4, max(1, len(thumbs)))
rows = (len(thumbs) + cols - 1) // cols
plt.figure(figsize=(4*cols, 4*rows))
for idx, (img_th, lbl) in enumerate(thumbs):
    plt.subplot(rows, cols, idx+1)
    plt.imshow(img_th)
    plt.title(lbl)
    plt.axis('off')
plt.suptitle('Face Analysis Results', fontsize=16)
plt.show()

**Tip:** `FaceAnalyzer` will download a FairFace ONNX model into the project's `models/` directory if not already present. This can take a minute on first run.